The work of the bake-off is **building** the binders; comparing them is a much smaller step. This notebook builds all four over the annual-report store --- it generates training samples from the store, fine-tunes the compiler (arm B), indexes those samples for the frozen few-shot binder (arm C), and calibrates the patchable alias table (arm D) --- and only then reads the persisted crossover. The build cells need the GPU and the licensed backend; run them locally (execution is off at render time).

In [ ]:
import sys, torch
if not torch.cuda.is_available():
    print(
        "This notebook requires a GPU (CUDA) — no GPU detected.\n"
        "Please re-run on a machine with a CUDA-capable GPU."
    )
    sys.exit(0)

**Setup.** Load the store and the base LM through the installed package.

In [ ]:
from forgeloop import data_path, ensure_artifacts, missing_artifacts
from forgeloop.rag import load_store

try:
    ensure_artifacts()      # fetch the store + adapters if missing (portable)
except FileNotFoundError:
    # ensure_artifacts() has no downloadable source for nl2triple (see
    # forgeloop.artifacts.PRODUCERS) and raises rather than fetching everything
    # else and reporting just that one. Anything besides nl2triple missing is a
    # real problem -- only the nl2triple-only case is ours to build below.
    if missing_artifacts() != ["nl2triple"]:
        raise

# nl2triple is a local GPU build, not a fetch: enrich_data.py generates the
# DoE test cohort (rag_cohort.json), make_data.py inverts it (plus a
# separately-seeded train split) into (question -> triple) pairs, and
# train_lora.py fine-tunes the adapter from them. Gated on the adapter's own
# config file rather than missing_artifacts(): make_data.py's os.makedirs
# creates data/nl2triple/ before doing any real work, so the manifest's bare
# directory-existence check would read "built" the moment data-gen starts,
# even if train_lora.py never finishes.
adapter_config = data_path("nl2triple", "qwen3_4b_triple_lora", "adapter_config.json")
if not adapter_config.exists():
    import subprocess
    import sys
    from pathlib import Path

    here = Path.cwd().resolve()
    scripts = next(
        p / "code" / "scripts"
        for p in (here, *here.parents)
        if (p / "code" / "scripts" / "nl2triple_experiment").is_dir())

    cohort = data_path("enrichment", "rag_cohort.json")
    if not cohort.exists():
        print("building enrichment: enrich_data.py ...")
        subprocess.run([sys.executable, str(scripts / "enrich_data.py")], check=True)

    for name in ("make_data.py", "train_lora.py"):
        print(f"building nl2triple: {name} ...")
        subprocess.run(
            [sys.executable, str(scripts / "nl2triple_experiment" / name)], check=True)

from knowlytix.knowledge.llm_backend import LocalTransformersBackend
from knowlytix.knowledge.rag.compiler import (build_compiler_dataset, train_compiler,
                                              CompilerSFTConfig, QWEN_4B)
store = load_store()        # forgeloop resolves the store dir -- no hardcoded path
llm = LocalTransformersBackend(QWEN_4B)   # rephraser for datagen + base for SFT

**Arm B, step 1 --- generate samples (a design of experiments).** `build_compiler_dataset` runs three knowlytix stages: graph generators mine base questions (single-hop, two-hop, and relation-absent probes), each carrying its gold hop chain as the training target; `DesignMatrix.from_catalog` draws a space-filling **Sobol** design over the ~20 presentation factors of the DoE (clarity, length, expertise, paraphrase depth, ...); and `QuestionRephraser` realizes each design row, rewriting the base question to those factor levels while preserving the target chain. A held-out factor level and a fraction of facts become the eval splits, so the design defines what the model is tested on.

In [ ]:
splits = build_compiler_dataset(
    store, llm, group='comprehensive', variants_per_base=12,
    heldout_levels={'clarity': 'Misleading'}, heldout_fact_frac=0.15, seed=42)
print({k: len(v) for k, v in splits.items()})
print('one sample:', splits['train'][0])

**Arm B, step 2 --- fine-tune the SLM compiler** on the samples with a low-rank adapter, written into the store's `query_compiler/`.

In [ ]:
compiler_dir = train_compiler(
    splits['train'], out_dir=str(data_path('gms_annual_report_store', 'query_compiler')),
    config=CompilerSFTConfig(base_model=QWEN_4B, lora_r=16, lora_alpha=32, epochs=3))
print('compiler adapter:', compiler_dir)

**Arm C --- index the same samples** in the tuned encoder's space (no training); the k nearest are shown to a frozen model at inference.

In [ ]:
from knowlytix.embedding import FineTunedEmbedding
from knowlytix.knowledge.rag.bakeoff import ExemplarIndex
v_encoder = FineTunedEmbedding.load(str(data_path('gms_annual_report_store', 'tuned_encoder')))
index = ExemplarIndex.from_rows(splits['train'], v_encoder.encode)
print('exemplars indexed:', len(index))

**Arm D --- build a patchable alias table** for the head entity, with a calibrated encoder-nearest fallback; a mis-binding is fixed by one edit.

In [ ]:
from knowlytix.knowledge.rag.bakeoff import AliasTableResolver, calibrate_alias_resolver
from knowlytix.knowledge.rag.compiler.walk import StoreChainWalker
entities = StoreChainWalker(store).entities
tau = calibrate_alias_resolver(entities, v_encoder.encode, far_ceiling=0.05)['tau']
resolver = AliasTableResolver.from_store(store, encoder=v_encoder.encode, tau=tau)
resolver.add_alias('CP', 'cloud platform')   # a patch is one row, not a retrain
# tau is None when the store cannot calibrate (<4 entities, or no encoder):
# the documented recusal path, which round() would turn into a TypeError.
print('alias entries:', len(resolver.table),
      '| fallback tau:', round(tau, 3) if tau is not None else 'disabled (recuses)')

**The comparison, in one step.** With the binders built, the crossover and the G4 verdict are read from the persisted run.

In [ ]:
import json
report = json.load(open(data_path('enrichment', 'bakeoff_ABCD.json')))
decision = json.load(open(data_path('enrichment', 'bakeoff_decision.json')))
for name, e in report['arms'].items():
    o = e['overall']
    print(f"{name:12} acc={o['accuracy']:.3f} mis_bind={o['mis_bind_rate']:.3f} "
          f"holdout={e['holdout']['accuracy']:.3f} patch={e['patch_cost']}")
print('decision:', decision['recommended'], '| recused:', decision['recused'])